## **Bibliotecas**

In [1]:
#%pip install pydantic[email]

import enum
import hashlib
import re
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    SecretStr,
    ValidationError,
)

In [2]:
# Garante uma senha forte.
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")

# ^          : Início da linha.
# (?=.*[a-z]): Lookahead positivo - deve conter pelo menos uma letra minúscula.
# (?=.*[A-Z]): Lookahead positivo - deve conter pelo menos uma letra maiúscula.
# (?=.*\d)   : Lookahead positivo - deve conter pelo menos um número.
# .{8,}      : Deve ter no mínimo 8 caracteres de qualquer tipo.


# Garante um nome simples e válido.
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

# ^          : Início da linha.
# [a-zA-Z]   : Permite apenas letras (maiúsculas ou minúsculas).
# {2,}       : Deve ter no mínimo 2 caracteres.

## **Classes**

In [3]:
class Role(enum.IntFlag):
    # Diferente do auto(), que gera os valores automaticamente, 
    # aqui os bits (1, 2, 4, 8) estão sendo definidos manualmente.

    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

In [4]:
class User(BaseModel):
    name: str = Field(examples=["Arjan"])

    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )

    password: SecretStr = Field(
        examples=["Password123"],
        description="The password of the user"
    )

    role: Role = Field(
        default=None,
        description="The role of the user",
        examples=[1, 2, 4, 8]
    )


    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        """
        Valida o nome usando o REGEX compilado anteriormente.
        """

        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

    # mode="before": A validação ocorre ANTES do Pydantic tentar converter o tipo.
    # Isso é útil para aceitar múltiplos formatos (int, str) e converter para Role.
    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        """
        Mapeia a entrada para o enum Role, aceitando o nome (str) ou o valor (int).
        Se a conversão falhar, lista as opções válidas para o usuário.
        """

        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

    @model_validator(mode="before")
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        """
        Valida regras complexas que dependem de mais de um campo e realiza
        o processamento final dos dados (como o Hashing da senha).
        """

        # Verificação se os campos existem
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")

        # Verificação se o nome do usuário está contido na senha
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")

        # Validação de força da senha via REGEX
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        
        # Transforma a senha em um Hash
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        
        return v

## **Funções**

In [5]:
def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid:")
        print(e)

## **Dados de Teste**

In [6]:
test_data = dict(
    good_data={
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "role": "Admin",
    },
    bad_role={
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "role": "Programmer",
    },
    bad_data={
        "name": "Arjan",
        "email": "bad email",
        "password": "bad password",
    },
    bad_name={
        "name": "Arjan<-_->",
        "email": "example@arjancodes.com",
        "password": "Password123",
    },
    duplicate={
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Arjan123",
    },
    missing_data={
        "email": "<bad data>",
        "password": "<bad data>",
    },
)

## **Main**

In [7]:
for example_name, data in test_data.items():
    print(example_name)
    validate(data)
    print()

good_data
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=<Role.Admin: 4>

bad_role
User is invalid:
1 validation error for User
role
  Value error, Role is invalid, please use one of the following: Author, Editor, Admin, SuperAdmin [type=value_error, input_value='Programmer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_data
User is invalid:
1 validation error for User
  Value error, Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number [type=value_error, input_value={'name': 'Arjan', 'email'...ssword': 'bad password'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_name
User is invalid:
1 validation error for User
name
  Value error, Name is invalid, must contain only letters and be at least 2 characters long [type=value_error, input_value='Arjan<-_->', input_type=str]
    For further information visit https://